In [1]:
from embedder import Embedder

# Inicializamos nuestro embedder ligero ONNX
embedder = Embedder()

# La pregunta de la Tarea 1
query = "How does approximate nearest neighbor search work?"

# Generamos el vector (embedding)
v = embedder.encode(query)

# Imprimimos el primer valor del vector
print(f"El primer valor (v[0]) es: {v[0]:.2f}")

El primer valor (v[0]) es: -0.02


In [2]:
from gitsource import GithubRepositoryDataReader

# 1. Cargamos los datos exactos del repositorio para la tarea
print("Descargando 72 documentos desde GitHub (esto tomará unos segundos)...")
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
print(f"Total de documentos descargados: {len(documents)}")

# 2. Buscamos el documento específico que nos pide la Q2
target_filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"
target_doc = next((doc for doc in documents if doc["filename"] == target_filename), None)

# 3. Vectorizamos su contenido y calculamos la similitud
if target_doc:
    # Convertimos el contenido (content) del documento a un vector
    doc_vector = embedder.encode(target_doc["content"])
    
    # Calculamos la Similitud del Coseno (Producto Punto) contra el vector 'v' de la Q1
    similitud_q2 = v.dot(doc_vector)
    
    print(f"\nResultado Q2 - Similitud del coseno: {similitud_q2:.2f}")
else:
    print("Error: No se encontró el documento.")

Descargando 72 documentos desde GitHub (esto tomará unos segundos)...
Total de documentos descargados: 72

Resultado Q2 - Similitud del coseno: 0.36


In [3]:
import numpy as np
from gitsource import chunk_documents

# 1. Fragmentamos los documentos en pedazos más pequeños
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Total de fragmentos (chunks) creados: {len(chunks)}")

# 2. Extraemos solo el texto (content) de cada chunk
chunk_contents = [chunk["content"] for chunk in chunks]

# 3. Vectorizamos TODOS los fragmentos para crear nuestra Matriz X
print("Vectorizando los fragmentos (esto tomará unos segundos)...")
# Usamos encode_batch que es más eficiente para procesar listas
X = np.array(embedder.encode_batch(chunk_contents))

# 4. Calculamos los puntajes multiplicando la matriz X contra el vector 'v' de la Q1
scores = X.dot(v)

# 5. Buscamos el índice (posición) con el puntaje más alto
best_idx = np.argmax(scores)
mejor_chunk = chunks[best_idx]

print("\n--- RESULTADO Q3 ---")
print(f"Puntaje de similitud más alto: {scores[best_idx]:.4f}")
print(f"El archivo al que pertenece este fragmento (filename) es:")
print(f"👉 {mejor_chunk['filename']}")

Total de fragmentos (chunks) creados: 295
Vectorizando los fragmentos (esto tomará unos segundos)...

--- RESULTADO Q3 ---
Puntaje de similitud más alto: 0.6489
El archivo al que pertenece este fragmento (filename) es:
👉 02-vector-search/lessons/07-sqlitesearch-vector.md


In [4]:
from minsearch import VectorSearch

# 1. Creamos la instancia de búsqueda vectorial en minsearch
# (Esta vez no necesitamos campos de filtro, solo el texto)
vindex = VectorSearch(keyword_fields=[])

# 2. Le pasamos nuestra matriz X y los fragmentos (chunks)
# Nota: Le pasamos 'chunks' en lugar de 'documents' completos porque 
# X fue creada a partir de los chunks.
vindex.fit(X, chunks)

# 3. La nueva pregunta de la tarea
query_q4 = "What metric do we use to evaluate a search engine?"
print(f"Pregunta: {query_q4}")

# 4. Convertimos la pregunta a vector
v_query_q4 = embedder.encode(query_q4)

# 5. Buscamos usando minsearch
resultados_q4 = vindex.search(
    v_query_q4,
    num_results=5
)

# 6. Imprimimos el primer resultado
primer_resultado = resultados_q4[0]
print("\n--- RESULTADO Q4 ---")
print("El archivo (filename) del primer resultado es:")
print(f"👉 {primer_resultado['filename']}")

Pregunta: What metric do we use to evaluate a search engine?

--- RESULTADO Q4 ---
El archivo (filename) del primer resultado es:
👉 04-evaluation/lessons/05-search-metrics.md


In [5]:
from minsearch import Index

# 1. Configuramos el motor de Texto Clásico (Palabras clave)
# Le decimos que busque dentro del campo "content"
tindex = Index(text_fields=["content"], keyword_fields=[])
tindex.fit(chunks)

# 2. La pregunta para la competencia
query_q5 = "How do I store vectors in PostgreSQL?"
print(f"Pregunta: {query_q5}\n")

# 3. BÚSQUEDA DE TEXTO (Palabras clave)
resultados_texto = tindex.search(query_q5, num_results=5)
archivos_texto = [doc['filename'] for doc in resultados_texto]

print("--- TOP 5: BÚSQUEDA DE TEXTO ---")
for f in archivos_texto: print(f"📄 {f}")

# 4. BÚSQUEDA VECTORIAL (Significado)
v_query_q5 = embedder.encode(query_q5)
resultados_vectoriales = vindex.search(v_query_q5, num_results=5)
archivos_vectoriales = [doc['filename'] for doc in resultados_vectoriales]

print("\n--- TOP 5: BÚSQUEDA VECTORIAL ---")
for f in archivos_vectoriales: print(f"📄 {f}")

# 5. La gran comparación: ¿Cuál está en vectores pero NO en texto?
print("\n--- RESULTADO Q5 ---")
for f in archivos_vectoriales:
    if f not in archivos_texto:
        print(f"👉 Este archivo SOLO lo encontró la Búsqueda Vectorial:")
        print(f"   {f}")

Pregunta: How do I store vectors in PostgreSQL?

--- TOP 5: BÚSQUEDA DE TEXTO ---
📄 02-vector-search/lessons/02-embeddings.md
📄 03-orchestration/lessons/05-rag.md
📄 02-vector-search/lessons/01-intro.md
📄 03-orchestration/lessons/05-rag.md
📄 02-vector-search/lessons/01-intro.md

--- TOP 5: BÚSQUEDA VECTORIAL ---
📄 02-vector-search/lessons/08-pgvector.md
📄 02-vector-search/lessons/08-pgvector.md
📄 03-orchestration/lessons/05-rag.md
📄 02-vector-search/lessons/08-pgvector.md
📄 02-vector-search/lessons/08-pgvector.md

--- RESULTADO Q5 ---
👉 Este archivo SOLO lo encontró la Búsqueda Vectorial:
   02-vector-search/lessons/08-pgvector.md
👉 Este archivo SOLO lo encontró la Búsqueda Vectorial:
   02-vector-search/lessons/08-pgvector.md
👉 Este archivo SOLO lo encontró la Búsqueda Vectorial:
   02-vector-search/lessons/08-pgvector.md
👉 Este archivo SOLO lo encontró la Búsqueda Vectorial:
   02-vector-search/lessons/08-pgvector.md


In [6]:
# 1. Definimos la función matemática RRF (tal como la da el profesor)
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    # Por cada lista de resultados (texto y vectores)...
    for results in result_lists:
        # enumerate() nos da dos cosas: la posición (rank) y el documento (doc)
        # OJO: En Python, el rank empieza a contar desde 0. 
        # (0 es el 1º lugar, 1 es el 2º lugar, etc.)
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            # Le damos puntos basados en su posición (rank). ¡Entre más arriba, mejor!
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    # Ordenamos los documentos por su nuevo puntaje híbrido
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

# 2. La pregunta final de la tarea
query_q6 = "How do I give the model access to tools?"
print(f"Pregunta: {query_q6}\n")

# 3. Obtenemos los resultados de AMBOS motores por separado
v_query_q6 = embedder.encode(query_q6)
vector_results = vindex.search(v_query_q6, num_results=5)
text_results = tindex.search(query_q6, num_results=5)

# 4. ¡Fusión Híbrida!
resultados_hibridos = rrf([vector_results, text_results])

# 5. Imprimimos al ganador absoluto
ganador = resultados_hibridos[0]
print("--- RESULTADO Q6 (BÚSQUEDA HÍBRIDA) ---")
print("El archivo (filename) clasificado en PRIMER LUGAR después de RRF es:")
print(f"👉 {ganador['filename']}")

Pregunta: How do I give the model access to tools?

--- RESULTADO Q6 (BÚSQUEDA HÍBRIDA) ---
El archivo (filename) clasificado en PRIMER LUGAR después de RRF es:
👉 01-agentic-rag/lessons/13-function-calling.md
